# Perfiles del personal: clustering multivista (SNF)

Integra tres vistas del perfil de cada persona —semántica de trayectoria y gestión, semántica académico-temática (embeddings `BAAI/bge-m3` de las evidencias) y estructurada (variables de los atributos)— con *Similarity Network Fusion*, y agrupa con clustering espectral en tres ámbitos. Método y salidas en `README.md` (DEC-045).

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "03_perfiles").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from perfiles import construir, embeddings

pd.set_option("display.max_colwidth", 120)

## 1. Embeddings de las evidencias (incremental: solo textos nuevos)

In [ ]:
manifest_emb = embeddings.construir()
manifest_emb

## 2. Vistas, SNF, clustering, interpretación y t-SNE

In [ ]:
manifest = construir.construir()
print(json.dumps(manifest['ambitos'], ensure_ascii=False, indent=2))

## 3. Selección del número de patrones y estabilidad

In [ ]:
carpeta = PROJECT_ROOT / "data" / "perfiles" / "clustering" / manifest["version"]
for ambito in ["todos", "administrativos", "docentes"]:
    r = json.loads((carpeta / ambito / "clusters.json").read_text(encoding="utf-8"))
    print(f"\n=== {r['nombre']}: k = {r['k']} ({r['seleccion_k']['criterio']}), estabilidad ARI = {r['estabilidad_ari']}, "
          f"perfiles mixtos = {r['perfiles_mixtos']}")
    display(pd.DataFrame(r["seleccion_k"]["tabla"]))

## 4. Patrones encontrados

In [ ]:
for ambito in ["todos", "administrativos", "docentes"]:
    r = json.loads((carpeta / ambito / "clusters.json").read_text(encoding="utf-8"))
    print(f"\n==================== {r['nombre']}")
    for c in r["clusters"]:
        print(f"\n[{c['cluster']}] {c['etiqueta']}  —  {c['tamano']} personas ({c['tamano_vigentes']} vigentes), "
              f"{c['perfiles_mixtos']} mixtos, representante: {c['representante']['cargo_actual']}")
        print("   ", c["descripcion"])
        print("    términos:", ", ".join(c["terminos_distintivos"][:8]))

## Segundo nivel: subpatrones

Cada patrón con al menos 150 personas se subdivide recalculando SNF solo con sus integrantes
(escalas de afinidad y estandarización locales) y clustering espectral con k por eigengap entre 2 y 8.
Los subpatrones se describen **frente a su patrón padre**, no frente al ámbito.

In [ ]:
for ambito in ["todos", "administrativos", "docentes"]:
    r = json.loads((carpeta / ambito / "clusters.json").read_text(encoding="utf-8"))
    print(f"\n==================== {r['nombre']}")
    for c in r["clusters"]:
        s = c.get("subdivision")
        if not s:
            print(f"\n[{c['cluster']}] {c['etiqueta']} ({c['tamano']} personas): no se subdivide")
            continue
        tabla = ", ".join(f"k={t['k']}: {t['eigengap']:.3f}" for t in s["seleccion_k"]["tabla"])
        print(f"\n[{c['cluster']}] {c['etiqueta']} ({c['tamano']} personas) -> {s['k']} subpatrones, "
              f"estabilidad ARI = {s['estabilidad_ari']}\n    eigengap: {tabla}")
        for f in s["subpatrones"]:
            print(f"    {f['subpatron_id']} {f['etiqueta']} · {f['tamano']} personas ({f['tamano_vigentes']} vigentes), "
                  f"{f['perfiles_mixtos']} mixtos\n         {f['descripcion']}\n"
                  f"         términos: {', '.join(f['terminos_distintivos'][:8])}")
